# MrVI

MrVI (Multi-Resolution Variational Inference) [P.Boyeau et al, 2025](https://www.nature.com/articles/s41592-025-02808-x) is a deep generative model designed for exploratory and comparative analysis of single-cell sequencing data from multiple samples

Takes as input two types of sample-level covariates:

- Target Covariates: (sample-level target covariates). Which refer to actual properties. It usually represents the sample ID which defined what entities will be compared. (In our case, they will be the sample ID). Given by: $s_n$

- Nuisance Covariates: Corresponds to technical factors (study of origin, sample processing site....) Given by: $b_n$



## MrVI process

![MrVI Graphical Model](images/MrVi.png)

Image taken from [scvi-tools](https://docs.scvi-tools.org/en/1.3.3/user_guide/models/mrvi.html)

The idea from MrVI is that each cell (n) is associated with two latent variables $u_n, z_n$

- **$u_n$**: Captures the variation between cell states without taking into account the sample covariates. Given by a Gaussian Mixture prior $u_n \sim MixtureOfGaussians(\mu_1, ..., \mu_k, \sum_1, ..., \sum_K, \pi_1, ..., \pi_K)$

- **$z_n$**: Captures the variation between cell states, taking into account the sample covariates but remaining unaffected by nuisance covariates. It is actually learnt from $u_n$ as well as the target covariates. $z_n|u_n \sim N(u_n, I_L)$. This allows for comparative analysis, as a useful question we could extract from this would be: "If cell n, originally in patient A (s), cambe from patient B (s'), how would the state $z_n$ look like". This is one of the main advantages of MrVI

- **$x_n$**: Models raw gene counts by decoding $z_n$ conditioned on nuisance covariates $b_n$ and cell library size $l_n$ using a Negative Binomial distribution:
  $$x_n \mid z_n, b_n \sim \text{NegativeBinomial}(\mu(z_n, b_n), r)$$

Actual $x_n$ is computed using $h_n$ which receives the latent $z_n$ and nuisance factors $b_n$ which pass through a multi-head attention layer so the output is normalized (this also uses a specific matrix and bias, more details in the paper)

## MrVI Inference

Variatonal inference is used to approximate the posterior of $u_n$ and $z_n$.

$q_\phi (u_n|x_n) = N(\mu_\phi (x_n), \sigma_\phi^2(x_n)I$

$z_n = u_n + f_\phi(u_n, s_n)$

Where the values are given by neural networks such as encoders or using multi-head attention

**Important: Model is already trained, if RETRAIN is set to FALSE above, this will load the already existing model**

In [ ]:
from scvi.external import MRVI

# Training MrVI
# Despite there being two separate latents which encode different information, they both do it in the same dimensions
def train_mrvi(adata, name="mrvi", sample_key="patient", batch_key=None, n_latent=N_LATENT, **model_kwargs):
    a = adata.copy()

    # Registers the dataset
    MRVI.setup_anndata(
        a, 
        layer="counts", 
        sample_key=sample_key, # Target covariate
        batch_key=batch_key # Nuisance factors 
    )

    path = MODELS_DIR / name
    if path.exists() and not RETRAIN:
        model = MRVI.load(path, adata=a)
    else:
        model = MRVI(a, n_latent=n_latent, **model_kwargs)
        model.train(max_epochs=400, early_stopping=True, check_val_every_n_epoch=1) # The tutorial uses 400 epochs, just in case, we also apply early stopping
        model.save(path, overwrite=True)

    return model, a

# Training MrVI
mrvi_model, mrvi_adata = train_mrvi(adata, name="mrvi_patient", sample_key="patient")

# Latent for each of the elements
adata.obsm["X_mrvi_u"] = mrvi_model.get_latent_representation(adata=mrvi_adata, give_z=False) # U latent
adata.obsm["X_mrvi_z"] = mrvi_model.get_latent_representation(adata=mrvi_adata, give_z=True) # Z latent


The ELBO loss for this model is given by:

$$
ELBO(\theta, \phi) = E_{x,s,b} E_{q_\phi(u|x)}[\log \frac{p_\theta(x|z,b) p_\theta(z|u,s) p_\theta(u)}{q_\phi(u|x)}]
$$

This is just for understanding purposes, it is also important to note that the final ELBO loss actually includes a penalty term to ensure that there is a distance between z and u so they do not represent the same element

Visualizing the ELBO loss to see for convergence or potential overfitting.

In [ ]:
h = mrvi_model.history
print("history keys:", list(h.keys()))   # confirm the validation key name
fig, ax = plt.subplots(figsize=(7, 4))
for key in ("elbo_train", "elbo_validation"):
    if key in h:
        ax.plot(h[key].values.ravel(), label=key)
ax.set(xlabel="epoch", ylabel="ELBO", title="MrVI convergence")
ax.legend()
plt.show()

In [ ]:
sc.pp.neighbors(adata, use_rep="X_mrvi_u", key_added="mrvi_u_neighbors")
sc.tl.umap(adata, neighbors_key="mrvi_u_neighbors")
adata.obsm["X_umap_mrvi_u"] = adata.obsm["X_umap"].copy()

sc.pp.neighbors(adata, use_rep="X_mrvi_z", key_added="mrvi_z_neighbors")
sc.tl.umap(adata, neighbors_key="mrvi_z_neighbors")
adata.obsm["X_umap_mrvi_z"] = adata.obsm["X_umap"].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sc.pl.embedding(
    adata, 
    basis="X_umap_mrvi_u", 
    color="celltype", 
    ax=axes[0, 0], 
    show=False, 
    title="u_n — Cell Types (Integrated)"
)
sc.pl.embedding(
    adata, 
    basis="X_umap_mrvi_u", 
    color="patient", 
    ax=axes[0, 1], 
    show=False, 
    title="u_n — Patients (Well Mixed)"
)

sc.pl.embedding(
    adata, 
    basis="X_umap_mrvi_z", 
    color="celltype", 
    ax=axes[1, 0], 
    show=False, 
    title="z_n — Cell Types"
)
sc.pl.embedding(
    adata, 
    basis="X_umap_mrvi_z", 
    color="patient", 
    ax=axes[1, 1], 
    show=False, 
    title="z_n — Patients (Sample Structure Preserved)"
)

plt.tight_layout()
plt.show()

We can see 4 separate graphs

First row displays the latent from u.

- Left Displays cell populations grouped into distinct clusters

- Right: Pattient labels

The main idea we can extract from here is that the latent space for $u_n$ clearly isolated cell identity without taking into account donor-specific covariates

Bottom row displays the latent from z.

- Left: Split by cell type, however, we can see that the main cells are split, dont show clear clusters as in the above one.

- Right: Align directly with patient IDs. Shows the effect of introducing target sample covariates $s_n$ in this case, patient ID which captures donor-to-donor variation

Cancer epithelial cells group together clearly for the latent space $u_n$. In $z_n$ cancer epithelial cells split into specific patients, showing variation across patients.

In [ ]:
y_true = adata.obs["celltype"].astype(str).values
n_celltypes = adata.obs["celltype"].nunique()

# 10000 cell subsample for Silhouette Score
np.random.seed(SEED)
sub_idx = np.random.choice(adata.shape[0], size=min(10000, adata.shape[0]), replace=False)

mrvi_results = []
for name, rep_key in [("MrVI (u_n)", "X_mrvi_u"), ("MrVI (z_n)", "X_mrvi_z")]:
    emb = adata.obsm[rep_key]

    kmeans = KMeans(n_clusters=n_celltypes, random_state=SEED, n_init=10)
    cluster_labels = kmeans.fit_predict(emb)
    
    ari = adjusted_rand_score(y_true, cluster_labels)
    nmi = normalized_mutual_info_score(y_true, cluster_labels)

    # ASW
    emb_sub = emb[sub_idx]
    asw_celltype = silhouette_score(emb_sub, adata.obs["celltype"].iloc[sub_idx])
    asw_patient = silhouette_score(emb_sub, adata.obs["patient"].iloc[sub_idx])

    batch_ent = compute_batch_entropy(emb, adata.obs["patient"].values, k=50)


    mrvi_results.append({
        "Model": name,
        "ARI": ari,
        "NMI": nmi,
        "ASW (Cell Type)": asw_celltype,
        "ASW (Patient)": asw_patient,
        "Batch Entropy": batch_ent,
    })

mrvi_metrics = pd.DataFrame(mrvi_results).set_index("Model")
print(mrvi_metrics)

As explained before, $u_n$ is a latent which is used for storing the variation between cell states. Therefore it consistenly outperforms $z_n$ for each of the cell-type metrics. Batch entropy is much higher showing that clearly this latent removes patient effects.

$z_n$ preserves patient-level variation. This explains the decline in batch entropy metrics as well as a much lower ASW (patient). This supports our UMAP representation above.

The first latent will be better for cell-type clustering as well as any task requiring elimination of batch noise. The second latent will be useful to study patient-specific problems if needed.

### Imports


In [2]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd()))   

from src import data as D       
from src import hvg as H        
from src import metrics as M    
from src import splits as S     
import numpy as np, pandas as pd

###  Load data and attach the folds

`load_adata()` finds `wu2021_brca.h5ad` automatically and fixes the layout: in the file `.X` is
already log1p-CP10k and the counts are in `.raw`, so after loading `adata.X` = **raw integer counts**
(what scVI/MrVI and seurat_v3 HVG need) and `adata.layers["lognorm"]` = log1p-CP10k (for PCA-like
baselines). Never apply `log1p` again. `attach_folds()` reads
`data/splits.parquet` (the 26-patient table from notebook 01) and maps each patient's
`fold_5 / fold_lopo / fold_batch` onto every cell via `donor_id`.

In [4]:
adata = D.load_adata()  # the real data: cells x genes
# D.attach_folds(adata) # adds obs['fold_5'], obs['fold_lopo'], obs['fold_batch']
# print(adata)
# adata.obs[['donor_id','fold_5','fold_lopo','fold_batch']].drop_duplicates('donor_id').head() # Safety head drop duplicates

In [5]:
D.attach_folds(adata)

AttributeError: 'NoneType' object has no attribute 'obs'

### 2. Fold

`fold_mask(adata, scheme, test_value)` returns boolean `train_mask, test_mask` with no cell
of a patient split across the two. We then select HVG on train only (`select_hvg_train`)
so feature selection never sees the test patient.

In [3]:
SCHEME, TEST_FOLD = "fold_5", 0
train_mask, test_mask = D.fold_mask(adata, SCHEME, TEST_FOLD)
print(f"{SCHEME}=={TEST_FOLD}:  train cells = {train_mask.sum():,}   test cells = {test_mask.sum():,}")

hvg = H.select_hvg_train(adata, train_mask, n_top=2000) # computed on TRAIN cells only
print("HVG selected on train:", len(hvg))

fold_5==0:  train cells = 78,344   test cells = 21,720


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


HVG selected on train: 2000


### 3. `### YOUR MODEL HERE`

Fit your model on the train cells / HVG only, then produce a latent embedding
(count models such as scVI use `adata.X`; Gaussian ones such as PCA use `layers["lognorm"]`)
`Z` for all cells. 
Below is a placeholder (PCA) just so the template runs end-to-end, but it can be replaced with other.

In [4]:
# ---- placeholder embedding ----
from sklearn.decomposition import PCA
X = adata[:, hvg].layers["lognorm"]   # already log1p-CP10k: no extra log
X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
Z_train = PCA(n_components=30, random_state=S.SEED).fit(X[train_mask])
Z = Z_train.transform(X)  # embedding for ALL cells (train fit, applied to all)
print("Z:", Z.shape)
# ------------------------------------------------------------------------

Z: (100064, 30)


### 4. Evaluate on the test fold

All metrics live in `src/metrics.py`, so all of us compute them the same way:
- `clustering_scores` -> ARI / NMI / cell-type silhouette (biological signal kept),
- `batch_entropy` -> patient mixing (batch correction worked).

# Evaluation Metrics

Evaluation will be done on a random 10.000 cell subsample for faster computation times. Despite it being randomly decided, a seed will be defined at each of this splits in order to ensure fairness across all the different methods.

As for the number of epochs, methods such as scVI use a heuristic approach for deciding it. For other methods we will simply apply a large number of epochs with Early Stopping to avoid potential overfitting.

The metrics we will use to evaluate the model will be applied at latent space level.They will be used to asses the quality of preserving cell-type identity as well as integrating patient samples. They are taken from some of the papers in the literature such as the Benchmarking paper [M.D. Luecken et. Al.](https://www.nature.com/articles/s41592-021-01336-8).
These are:

- **[Adjusted Rand Index (ARI)](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html)**:  Evaluates the similarity between two clustering assignments. Takes the Rand Index (RI) which computes the percentage of correct decisions comparing the predicted clusters to the true labels. In our case, we compare to the true cell-type labels. However, random assignments could have positive values, which is addressed by Adjusted Rand Index. Returns a value between -1 and 1. 1 means perfect classification, 0 corresponds to random cluster assignment

- **[Normalized Mutual Information (NMI)](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.normalized_mutual_info_score.html)**: Mutual Information splits according to clusters and according to class labels. Then, it quantifies how these two splittings agree with each other. However, MI gets large if the number of clusters is large. Therefore, this number is normalized to be between 0 (no mutual information) and 1 (perfect correlation). The higher the better

- **[Average Silhouette Width (ASW)](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)**: Measures how close an element is to its own cluster while comparing to its other clusters. Returns a value between -1 and 1, where negative values will correspond to when a sample has been assigned to the wrong cluster, positive values indicate good clustering. Values close to 0 are for overlapping clusters.

    The intepretation for this value depends on whether if we want to inspect quality of preserving cell-type identity or integrating patient samples.
    *ASW (cell):* The grouping variable will be the cell-type, we want this value to be high as cells of the same type should form isolated clusters in latent space.
    *ASW(patient):* If this value is high, it means that cells are clustering by patient rather than by biology elements.Therefore we want values closer to 0, as patients contain technical effects, which are usually unwanted (technical effects could be being inspected by a different doctor, different hospital, different times...)

- **Batch Entropy**: Measures the local mixing of a label in the k-nearest neighbor (kNN) graph around each cell. For every cell, the algorithm creates a neighborhood of its k nearest neighbors. (In the below function we use k=50, this will be the same for all models). A low value means that a cell neighborhood has cells from a single label.This can be used to compute for patients -> 'labels=adata.obs["batch_run"]' or 'labels=adata.obs["batch_run"]' are some of the possible options. 

    Our current batch entropy evaluates K=50 neighbors, as we have less number of patients and batch runs, is is possible that the theoretical maximum is slightly lower than 1, we will compute it so we can use it for reference when we obtain values.

In [ ]:
te = test_mask
scores = M.evaluate_embedding(
    Z=Z[te],
    cell_type=adata.obs["celltype_major"].values[te],
    patient=adata.obs["donor_id"].values[te],
    batch_run=adata.obs["batch_run"].values[te],
    k=50,
    seed=S.SEED,
)
print("test fold", TEST_FOLD, "->", {k: round(v, 3) for k, v in scores.items()})

test fold 0 -> {'ARI': 0.752, 'NMI': 0.807, 'ASW_celltype': 0.643} | patient_mixing: 0.225


### 5. Loop over every fold (cross-validation)

`iter_folds` yields `(test_value, train_mask, test_mask)` for the whole scheme. 

In [ ]:
rows = []
for test_value, tr, te in D.iter_folds(adata, scheme="fold_5"):
    hvg_k = H.select_hvg_train(adata, tr, n_top=2000)
    Xk = adata[:, hvg_k].layers["lognorm"]
    Xk = Xk.toarray() if hasattr(Xk, "toarray") else np.asarray(Xk)
    Zk = PCA(n_components=30, random_state=S.SEED).fit(Xk[tr]).transform(Xk)  # <-- your model here too

    s = M.evaluate_embedding(
        Z=Zk[te],
        cell_type=adata.obs["celltype_major"].values[te],
        patient=adata.obs["donor_id"].values[te],
        batch_run=adata.obs["batch_run"].values[te],
        k=50,
        seed=S.SEED,
    )
    s["fold"] = test_value
    rows.append(s)

res = pd.DataFrame(rows).set_index("fold")
print(res.round(3))
print("\nMEAN:", res.mean(numeric_only=True).round(3).to_dict())
print("STD :", res.std(numeric_only=True).round(3).to_dict())

        ARI    NMI  ASW_celltype  patient_mixing
fold                                            
3     0.538  0.696         0.625           0.252
2     0.544  0.709         0.638           0.138
4     0.677  0.786         0.666           0.158
1     0.632  0.743         0.657           0.188
0     0.752  0.807         0.643           0.225

MEAN: {'ARI': 0.629, 'NMI': 0.748, 'ASW_celltype': 0.646, 'patient_mixing': 0.192}


### Conclusion
Copy this notebook, replace the two `### YOUR MODEL HERE` boxes with your model,
and we get comparable and leak free CV metrics automatically because the splits, HVG rule and
metrics all come from the shared `src/`.